In [1]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("../data/processed/Ambazari_Nagpur_Integrated_2024_2025.csv")

In [4]:
data["timestamp"] = pd.to_datetime(data["timestamp"])
data = data.sort_values("timestamp").reset_index(drop=True)

In [6]:
print("Shape:", data.shape)

print("\nDate range:")
print(data["timestamp"].min(), "→", data["timestamp"].max())

print("\nColumns:")
print(data.columns.tolist())

Shape: (17544, 12)

Date range:
2024-01-01 00:00:00 → 2025-12-31 23:00:00

Columns:
['timestamp', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'wind_direction', 'weather_code']


In [ ]:
# Time-based features

data["hour"] = data["timestamp"].dt.hour
data["day"] = data["timestamp"].dt.day
data["day_of_week"] = data["timestamp"].dt.dayofweek
data["month"] = data["timestamp"].dt.month
data["year"] = data["timestamp"].dt.year

# Weekend indicator
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

In [8]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"


data["season"] = data["month"].apply(get_season)

In [9]:
print(data[[ "timestamp", "hour", "day", "day_of_week", "month", "year", "is_weekend", "season"]].head(10))

            timestamp  hour  day  day_of_week  month  year  is_weekend  season
0 2024-01-01 00:00:00     0    1            0      1  2024           0  Winter
1 2024-01-01 01:00:00     1    1            0      1  2024           0  Winter
2 2024-01-01 02:00:00     2    1            0      1  2024           0  Winter
3 2024-01-01 03:00:00     3    1            0      1  2024           0  Winter
4 2024-01-01 04:00:00     4    1            0      1  2024           0  Winter
5 2024-01-01 05:00:00     5    1            0      1  2024           0  Winter
6 2024-01-01 06:00:00     6    1            0      1  2024           0  Winter
7 2024-01-01 07:00:00     7    1            0      1  2024           0  Winter
8 2024-01-01 08:00:00     8    1            0      1  2024           0  Winter
9 2024-01-01 09:00:00     9    1            0      1  2024           0  Winter


In [10]:
# PM2.5 lag features

data["pm25_lag_1"] = data["pm25"].shift(1)
data["pm25_lag_3"] = data["pm25"].shift(3)
data["pm25_lag_6"] = data["pm25"].shift(6)
data["pm25_lag_12"] = data["pm25"].shift(12)
data["pm25_lag_24"] = data["pm25"].shift(24)

In [12]:
print(data[["timestamp", "pm25", "pm25_lag_1", "pm25_lag_3", "pm25_lag_6", "pm25_lag_12", "pm25_lag_24"]].head(30).T)

                              0                    1                    2   \
timestamp    2024-01-01 00:00:00  2024-01-01 01:00:00  2024-01-01 02:00:00   
pm25                   47.724032            47.724032            47.724032   
pm25_lag_1                   NaN            47.724032            47.724032   
pm25_lag_3                   NaN                  NaN                  NaN   
pm25_lag_6                   NaN                  NaN                  NaN   
pm25_lag_12                  NaN                  NaN                  NaN   
pm25_lag_24                  NaN                  NaN                  NaN   

                              3                    4                    5   \
timestamp    2024-01-01 03:00:00  2024-01-01 04:00:00  2024-01-01 05:00:00   
pm25                   47.724032            47.724032               108.17   
pm25_lag_1             47.724032            47.724032            47.724032   
pm25_lag_3             47.724032            47.724032          

In [13]:
pollutants = ["pm10", "no2", "so2", "co", "o3"]
lag_hours = [1, 6, 24]

for pollutant in pollutants:
    for lag in lag_hours:
        data[f"{pollutant}_lag_{lag}"] = data[pollutant].shift(lag)

In [ ]:
lag_columns = [column for column in data.columns if "_lag_" in column]

print("Number of lag features:", len(lag_columns))
print(lag_columns)

Number of lag features: 20
['pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_12', 'pm25_lag_24', 'pm10_lag_1', 'pm10_lag_6', 'pm10_lag_24', 'no2_lag_1', 'no2_lag_6', 'no2_lag_24', 'so2_lag_1', 'so2_lag_6', 'so2_lag_24', 'co_lag_1', 'co_lag_6', 'co_lag_24', 'o3_lag_1', 'o3_lag_6', 'o3_lag_24']


In [15]:
rolling_windows = [3, 6, 12, 24]

for window in rolling_windows:
    data[f"pm25_rolling_mean_{window}"] = (data["pm25"].shift(1).rolling(window=window).mean())

print(data[["timestamp", "pm25", "pm25_rolling_mean_3", "pm25_rolling_mean_6", "pm25_rolling_mean_12", "pm25_rolling_mean_24"]].tail(10))

                timestamp   pm25  pm25_rolling_mean_3  pm25_rolling_mean_6  \
17534 2025-12-31 14:00:00  71.29            81.236667            85.398333   
17535 2025-12-31 15:00:00  67.11            75.550000            82.620000   
17536 2025-12-31 16:00:00  64.03            71.353333            78.391667   
17537 2025-12-31 17:00:00  70.07            67.476667            74.356667   
17538 2025-12-31 18:00:00  51.18            67.070000            71.310000   
17539 2025-12-31 19:00:00  54.52            61.760000            66.556667   
17540 2025-12-31 20:00:00  70.53            58.590000            63.033333   
17541 2025-12-31 21:00:00  93.49            58.743333            62.906667   
17542 2025-12-31 22:00:00  95.40            72.846667            67.303333   
17543 2025-12-31 23:00:00  90.88            86.473333            72.531667   

       pm25_rolling_mean_12  pm25_rolling_mean_24  
17534             90.635833             91.090833  
17535             87.520833          

In [ ]:
pollutants = ["pm10", "no2", "so2", "co", "o3"]
rolling_windows = [3, 6, 24]

for pollutant in pollutants:
    for window in rolling_windows:
        data[f"{pollutant}_rolling_mean_{window}"] = (data[pollutant].shift(1).rolling(window=window).mean())

In [17]:
rolling_columns = [column for column in data.columns if "rolling_mean" in column]

print("Number of rolling features:", len(rolling_columns))
print(rolling_columns)

Number of rolling features: 19
['pm25_rolling_mean_3', 'pm25_rolling_mean_6', 'pm25_rolling_mean_12', 'pm25_rolling_mean_24', 'pm10_rolling_mean_3', 'pm10_rolling_mean_6', 'pm10_rolling_mean_24', 'no2_rolling_mean_3', 'no2_rolling_mean_6', 'no2_rolling_mean_24', 'so2_rolling_mean_3', 'so2_rolling_mean_6', 'so2_rolling_mean_24', 'co_rolling_mean_3', 'co_rolling_mean_6', 'co_rolling_mean_24', 'o3_rolling_mean_3', 'o3_rolling_mean_6', 'o3_rolling_mean_24']


In [19]:
# Convert wind direction from degrees to radians
wind_direction_rad = np.deg2rad(data["wind_direction"])

# Circular encoding
data["wind_direction_sin"] = np.sin(wind_direction_rad)
data["wind_direction_cos"] = np.cos(wind_direction_rad)
print(data[["wind_direction", "wind_direction_sin", "wind_direction_cos"]].head(10))

   wind_direction  wind_direction_sin  wind_direction_cos
0             177        5.233596e-02           -0.998630
1              11        1.908090e-01            0.981627
2              31        5.150381e-01            0.857167
3             360       -2.449294e-16            1.000000
4               7        1.218693e-01            0.992546
5              33        5.446390e-01            0.838671
6              28        4.694716e-01            0.882948
7              40        6.427876e-01            0.766044
8              66        9.135455e-01            0.406737
9              86        9.975641e-01            0.069756


In [ ]:
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)

data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

In [ ]:
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)

data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

In [25]:
pollutants = ["pm25", "pm10", "no2", "so2", "co", "o3"]

for pollutant in pollutants:
    data[f"target_{pollutant}"] = data[pollutant].shift(-1)

In [26]:
print(
    data[[
            "timestamp",
            "pm25", "target_pm25",
            "pm10", "target_pm10",
            "no2", "target_no2",
            "so2", "target_so2",
            "co", "target_co",
            "o3", "target_o3"
        ]].head(10))

            timestamp        pm25  target_pm25        pm10  target_pm10  \
0 2024-01-01 00:00:00   47.724032    47.724032   98.270973    98.270973   
1 2024-01-01 01:00:00   47.724032    47.724032   98.270973    98.270973   
2 2024-01-01 02:00:00   47.724032    47.724032   98.270973    98.270973   
3 2024-01-01 03:00:00   47.724032    47.724032   98.270973    98.270973   
4 2024-01-01 04:00:00   47.724032   108.170000   98.270973   172.200000   
5 2024-01-01 05:00:00  108.170000   102.100000  172.200000   158.400000   
6 2024-01-01 06:00:00  102.100000    88.310000  158.400000   147.010000   
7 2024-01-01 07:00:00   88.310000    84.490000  147.010000   157.550000   
8 2024-01-01 08:00:00   84.490000    83.770000  157.550000   163.400000   
9 2024-01-01 09:00:00   83.770000    92.440000  163.400000   174.280000   

         no2  target_no2        so2  target_so2        co  target_co  \
0  22.680115   22.680115  16.855077   16.855077  0.692535   0.692535   
1  22.680115   22.680115  16.8

In [ ]:
print(
    data[[
            "timestamp",
            "target_pm25",
            "target_pm10",
            "target_no2",
            "target_so2",
            "target_co",
            "target_o3"
        ]].tail())

                timestamp  target_pm25  target_pm10  target_no2  target_so2  \
17539 2025-12-31 19:00:00        70.53       115.78       81.59       22.91   
17540 2025-12-31 20:00:00        93.49       142.74       54.34       18.64   
17541 2025-12-31 21:00:00        95.40       144.01       52.45       13.36   
17542 2025-12-31 22:00:00        90.88       135.49       43.53       11.75   
17543 2025-12-31 23:00:00          NaN          NaN         NaN         NaN   

       target_co  target_o3  
17539       2.50   4.500000  
17540       1.46   8.610000  
17541       2.05   6.115046  
17542       2.55   0.000000  
17543        NaN        NaN  


In [28]:
print(data.isnull().sum().sort_values(ascending=False).head(30))

pm10_rolling_mean_24    24
pm25_lag_24             24
co_lag_24               24
pm10_lag_24             24
no2_lag_24              24
so2_lag_24              24
no2_rolling_mean_24     24
o3_rolling_mean_24      24
co_rolling_mean_24      24
so2_rolling_mean_24     24
pm25_rolling_mean_24    24
o3_lag_24               24
pm25_rolling_mean_12    12
pm25_lag_12             12
pm10_lag_6               6
co_lag_6                 6
o3_rolling_mean_6        6
pm10_rolling_mean_6      6
pm25_rolling_mean_6      6
so2_rolling_mean_6       6
pm25_lag_6               6
no2_lag_6                6
so2_lag_6                6
o3_lag_6                 6
no2_rolling_mean_6       6
co_rolling_mean_6        6
pm25_lag_3               3
no2_rolling_mean_3       3
pm25_rolling_mean_3      3
pm10_rolling_mean_3      3
dtype: int64


In [35]:
final_model_data = data.dropna().reset_index(drop=True)

In [36]:
print("Original rows:", len(data))
print("Model-ready rows:", len(final_model_data))
print("Rows removed:", len(data) - len(final_model_data))

Original rows: 17544
Model-ready rows: 17519
Rows removed: 25


In [37]:
print("Remaining missing values:", final_model_data.isnull().sum().sum())
print("Duplicate timestamps:", final_model_data["timestamp"].duplicated().sum())

Remaining missing values: 0
Duplicate timestamps: 0


In [38]:
final_model_data.to_csv(
    "../data/processed/Ambazari_Nagpur_Final_Model_Dataset_2024_2025.csv",
    index=False
)